## Buscador de Vibras

## Primer paso: Scraping de Citas al JSON

In [12]:
import json
import os
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Nombre del archivo que actuará como nuestra base de datos local rápida
JSON_FILE = "citas_cache.json"
citas_extraidas = []

# --- COMPLEMENTO DE PERSISTENCIA ---
# Verificamos si ya descargamos los datos previamente para evitar doble trabajo
if os.path.exists(JSON_FILE):
    print(
        f"📦 Carga local: Leyendo citas desde '{JSON_FILE}' sin hacer peticiones web..."
    )
    with open(JSON_FILE, "r", encoding="utf-8") as f:
        citas_extraidas = json.load(f)
else:
    print("🌐 Cache vacía: Conectando a la web para extraer citas...")
    URL = "https://quotes.toscrape.com/"
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        }
        response = requests.get(URL, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        bloques_quotes = soup.find_all("div", class_="quote")

        for bloque in bloques_quotes:
            elemento_texto = bloque.find("span", class_="text")
            elemento_autor = bloque.find("small", class_="author")

            if elemento_texto and elemento_autor:
                cita_limpia = (
                    elemento_texto.get_text()
                    .strip()
                    .replace("“", "")
                    .replace("”", "")
                )
                autor_limpio = elemento_autor.get_text().strip()
                citas_extraidas.append(
                    {"cita": cita_limpia, "autor": autor_limpio}
                )

        if citas_extraidas:
            # Guardamos físicamente en el disco duro del proyecto
            with open(JSON_FILE, "w", encoding="utf-8") as f:
                json.dump(citas_extraidas, f, ensure_ascii=False, indent=4)
            print(f"💾 Éxito: Base de datos local guardada en '{JSON_FILE}'")
        else:
            raise ValueError("No se encontraron estructuras válidas.")

    except Exception as error_scraping:
        print(f"⚠️ Error en red: {error_scraping}. Cargando respaldo...")
        citas_extraidas = [
            {
                "cita": "The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.",
                "autor": "Albert Einstein",
            },
            {
                "cita": "It is our choices, Harry, that show what we truly are, far more than our abilities.",
                "autor": "J.K. Rowling",
            },
        ]

# Creamos el DataFrame exactamente igual para que el resto del script funcione
df_citas = pd.DataFrame(citas_extraidas)
print(f"📊 Buscador listo: {len(df_citas)} frases cargadas en memoria.")


🌐 Cache vacía: Conectando a la web para extraer citas...
💾 Éxito: Base de datos local guardada en 'citas_cache.json'
📊 Buscador listo: 10 frases cargadas en memoria.


In [13]:
import json
import os
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup

JSON_FILE = "citas_cache.json"
citas_extraidas = []

# --- COMPLEMENTO DE PERSISTENCIA CON PAGINACIÓN COMPLETA ---
if os.path.exists(JSON_FILE):
    print(
        f"📦 Carga local: Leyendo citas desde '{JSON_FILE}' sin hacer peticiones web..."
    )
    with open(JSON_FILE, "r", encoding="utf-8") as f:
        citas_extraidas = json.load(f)
else:
    print("🌐 Cache vacía: Iniciando Web Scraping iterativo multihonda...")

    # Configuramos la URL base y las credenciales de navegación básicas
    BASE_URL = "https://toscrape.com"
    ruta_actual = "/"  # Empezamos en la raíz del sitio
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    pagina_contador = 1

    try:
        # El ciclo continuará ejecutándose mientras existan páginas consecutivas vinculadas
        while ruta_actual:
            url_destino = f"{BASE_URL}{ruta_actual}"
            print(f"   📄 Escaneando e indexando: Página {pagina_contador}...")

            response = requests.get(url_destino, headers=headers, timeout=15)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, "html.parser")
            bloques_quotes = soup.find_all("div", class_="quote")

            # Procesamiento de elementos dentro de la página actual
            for bloque in bloques_quotes:
                elemento_texto = bloque.find("span", class_="text")
                elemento_autor = bloque.find("small", class_="author")

                if elemento_texto and elemento_autor:
                    cita_limpia = (
                        elemento_texto.get_text()
                        .strip()
                        .replace("“", "")
                        .replace("”", "")
                    )
                    autor_limpio = elemento_autor.get_text().strip()
                    citas_extraidas.append(
                        {"cita": cita_limpia, "autor": autor_limpio}
                    )

            # --- DETECCIÓN DINÁMICA DE LA SIGUIENTE PÁGINA ---
            # Buscamos la etiqueta de lista de paginación que contiene el enlace "Next"
            elemento_next = soup.find("li", class_="next")

            if elemento_next:
                elemento_link = elemento_next.find("a")
                if elemento_link and elemento_link.get("href"):
                    ruta_actual = elemento_link["href"]  # Guardamos la ruta
                    pagina_contador += 1
                else:
                    ruta_actual = None
            else:
                # Si no existe la clase 'next', significa que llegamos al final del catálogo
                ruta_actual = None

        if citas_extraidas:
            # Guardado definitivo de todo el catálogo mapeado en el disco
            with open(JSON_FILE, "w", encoding="utf-8") as f:
                json.dump(citas_extraidas, f, ensure_ascii=False, indent=4)
            print(
                f"\n💾 ¡Éxito de Persistencia! Base de datos guardada en '{JSON_FILE}'"
            )
        else:
            raise ValueError(
                "La extracción iterativa finalizó sin recolectar registros."
            )

    except Exception as error_paginacion:
        print(
            f"\n⚠️ Falla durante el recorrido en la página {pagina_contador}: {error_paginacion}"
        )
        print("Activando base de datos de respaldo parcial...")
        citas_extraidas = [
            {
                "cita": "The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.",
                "autor": "Albert Einstein",
            },
            {
                "cita": "It is our choices, Harry, that show what we truly are, far more than our abilities.",
                "autor": "J.K. Rowling",
            },
        ]

# Reconstrucción idéntica del DataFrame de consumo para el motor semántico
df_citas = pd.DataFrame(citas_extraidas)
print(
    f"📊 Buscador optimizado listo: {len(df_citas)} frases cargadas en memoria."
)


📦 Carga local: Leyendo citas desde 'citas_cache.json' sin hacer peticiones web...
📊 Buscador optimizado listo: 10 frases cargadas en memoria.


Segundo Paso: Lee las frases de Quote to Scraping

In [14]:
import json
import os
import re
import sys
import ipywidgets as widgets
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import clear_output, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Configuración del archivo de persistencia local
JSON_FILE = "citas_cache.json"
citas_extraidas = []

# --- PASO 1: WEB SCRAPING CON PERSISTENCIA Y PAGINACIÓN COMPLETA ---
if os.path.exists(JSON_FILE):
    print(
        f"📦 Carga local: Leyendo citas desde '{JSON_FILE}' sin hacer peticiones web..."
    )
    with open(JSON_FILE, "r", encoding="utf-8") as f:
        citas_extraidas = json.load(f)
else:
    print("🌐 Cache vacía: Iniciando Web Scraping iterativo multihonda...")

    BASE_URL = "https://quotes.toscrape.com"
    ruta_actual = "/"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    pagina_contador = 1

    try:
        while ruta_actual:
            url_destino = f"{BASE_URL}{ruta_actual}"
            print(f"   📄 Escaneando e indexando: Página {pagina_contador}...")

            response = requests.get(url_destino, headers=headers, timeout=15)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, "html.parser")
            bloques_quotes = soup.find_all("div", class_="quote")

            for bloque in bloques_quotes:
                elemento_texto = bloque.find("span", class_="text")
                elemento_autor = bloque.find("small", class_="author")

                if elemento_texto and elemento_autor:
                    cita_limpia = (
                        elemento_texto.get_text()
                        .strip()
                        .replace("“", "")
                        .replace("”", "")
                    )
                    autor_limpio = elemento_autor.get_text().strip()
                    citas_extraidas.append(
                        {"cita": cita_limpia, "autor": autor_limpio}
                    )

            # Detección del botón "Next" para avanzar de página automáticamente
            elemento_next = soup.find("li", class_="next")
            if elemento_next:
                elemento_link = elemento_next.find("a")
                if elemento_link and elemento_link.get("href"):
                    ruta_actual = elemento_link["href"]
                    pagina_contador += 1
                else:
                    ruta_actual = None
            else:
                ruta_actual = None

        if citas_extraidas:
            with open(JSON_FILE, "w", encoding="utf-8") as f:
                json.dump(citas_extraidas, f, ensure_ascii=False, indent=4)
            print(
                f"\n💾 ¡Éxito de Persistencia! Base de datos guardada en '{JSON_FILE}'"
            )
        else:
            raise ValueError(
                "La extracción iterativa finalizó sin recolectar registros."
            )

    except Exception as error_paginacion:
        print(
            f"\n⚠️ Falla durante el recorrido en la página {pagina_contador}: {error_paginacion}"
        )
        print("Activando base de datos de respaldo parcial...")
        citas_extraidas = [
            {
                "cita": "The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.",
                "autor": "Albert Einstein",
            },
            {
                "cita": "It is our choices, Harry, that show what we truly are, far more than our abilities.",
                "autor": "J.K. Rowling",
            },
        ]

# Conversión al DataFrame requerida por los siguientes componentes
df_citas = pd.DataFrame(citas_extraidas)
print(
    f"📊 Buscador optimizado listo: {len(df_citas)} frases totales cargadas en memoria."
)

# --- PASO 2: MATRIZ DE SEMÁNTICA ABSTRACTA (SUB-PALABRAS) ---
vectorizador = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), lowercase=True
)
matrices_citas = vectorizador.fit_transform(df_citas["cita"])


# --- PASO 3: CONTROL DE INTENCIONES Y MANEJO DE ERRORES ---
def procesar_situacion_personal(intervencion_usuario):
    if not intervencion_usuario or not isinstance(intervencion_usuario, str):
        raise ValueError("El espacio de texto no puede ser procesado si está vacío.")

    entrada_limpia = intervencion_usuario.strip()

    if len(entrada_limpia) < 15:
        raise ValueError(
            "Por favor, detalla un poco más tu situación (mínimo 15 caracteres para análisis semántico)."
        )

    if entrada_limpia.isnumeric():
        raise ValueError(
            "Entrada inválida. Describe tu situación emocional con palabras, no con números."
        )

    matriz_usuario = vectorizador.transform([entrada_limpia])
    puntuaciones_similitud = cosine_similarity(
        matriz_usuario, matrices_citas
    ).flatten()

    indices_top = puntuaciones_similitud.argsort()[-3:][::-1]

    return [
        {
            "cita": df_citas.iloc[idx]["cita"],
            "autor": df_citas.iloc[idx]["autor"],
            "afinidad": puntuaciones_similitud[idx],
        }
        for idx in indices_top
    ]


# --- PASO 4: INTERFAZ INTERACTIVA EN JUPYTER ---
txt_situacion = widgets.Textarea(
    value="",
    placeholder="Ej: Siento que he cometido muchos errores seguidos y me cuesta volver a intentarlo...",
    description="Tu Situación:",
    layout=widgets.Layout(width="85%", height="110px"),
)

btn_vincular = widgets.Button(
    description="Analizar Situación",
    button_style="primary",
    icon="brain",
    layout=widgets.Layout(width="220px"),
)

out_pantalla = widgets.Output()


def ejecutar_interaccion(b):
    with out_pantalla:
        clear_output()
        try:
            afiliaciones = procesar_situacion_personal(txt_situacion.value)
            print("🧠 Análisis Semántico de Contexto Terminado\n")
            print(
                "Las 3 reflexiones más cercanas conceptualmente a tu vivencia actual son:\n"
            )
            print("=" * 75)
            for i, elemento in enumerate(afiliaciones, 1):
                print(
                    f"\n📌 Cita Vinculada #{i} (Afinidad Semántica: {elemento['afinidad']:.2%})"
                )
                print(f"«{elemento['cita']}»")
                print(f"👉 Perspectiva de: {elemento['autor']}\n")
                print("-" * 75)
        except ValueError as error_validacion:
            print(f"⚠️ Validación: {error_validacion}")
        except Exception as e:
            print(f"❌ Error inesperado: {e}")


btn_vincular.on_click(ejecutar_interaccion)
display(widgets.VBox([txt_situacion, btn_vincular, out_pantalla]))


📦 Carga local: Leyendo citas desde 'citas_cache.json' sin hacer peticiones web...
📊 Buscador optimizado listo: 10 frases totales cargadas en memoria.
